# Phase 2D (Kaggle GPU): Full 6 Feature Sets × 5 Seeds (80/20 Split, N_test = 542)

Notebook này chạy trọn vẹn **60 lượt Walk-Forward Residual (`6 FS × Expanding & Rolling × 5 seeds 41–45`)** tại mốc **80/20 (`542` phiên test)** theo đúng 100% giao thức của `copper_rolling_expanding_quan.ipynb`.

### Hướng dẫn chạy trên Kaggle (~20–30 phút với GPU T4/P100):
1. **Add Data:** Gắn dataset chứa file `dataset_9_commodities_daily.csv`.
2. **Accelerator:** Bật `GPU T4 x2` hoặc `GPU P100` (hoặc `GPU T4`).
3. Nhấn **Run All** — mỗi lượt chạy xong tự lưu checkpoint vào `/kaggle/working/ckpt_80/`.
4. Chạy xong tải tệp **`phase2d_80_results.zip`** từ `/kaggle/working/` về rồi gửi lại để gộp chung với bộ `90/10` (`60/60`) đã chạy xong.

In [1]:
from __future__ import annotations

import gc
import math
import os
import random
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler

# === CẤU HÌNH ĐỒNG BỘ 100% VỚI NOTEBOOK CHÍNH ===
MODE = "FULL_RESEARCH_MODE"
SEED = 42
SEEDS_MULTI = [41, 42, 43, 44, 45]
LOOKBACK = 22
HORIZON = 1
TARGET_RATIO = 0.80
VALIDATION_FRACTION = 0.10

BEST_W = 756
BEST_S = 40
BEST_UPDATE_MODE = "cold_start"

FS01_CU = ["Copper"]
FEATURES_4 = ["Copper", "WTI", "Gold", "Silver"]
FEATURES_3 = ["WTI", "Gold", "Silver"]
FS04_METALS_CU = ["Copper", "Aluminium", "Tin", "Silver", "Gold"]
FS05_METALS_EXO = ["Aluminium", "Tin", "Silver", "Gold"]
FS06_WTI_CU = ["Copper", "Aluminium", "Tin", "WTI"]
CANDIDATES = ["Aluminium", "Nickel", "Tin", "Iron_Ore_62", "Silver", "Gold", "WTI", "Brent"]

FEATURE_LIST = [
    ("FS01_CU", FS01_CU),
    ("FEATURES_4", FEATURES_4),
    ("FEATURES_3", FEATURES_3),
    ("FS04_METALS_CU", FS04_METALS_CU),
    ("FS05_METALS_EXO", FS05_METALS_EXO),
    ("FS06_WTI_CU", FS06_WTI_CU),
]
STRATEGIES = ["expanding_window", "rolling_window"]

PAPER_CONFIG = {"lstm_units_1": 39, "lstm_units_2": 111, "learning_rate": 5e-5}
ACTIVE_HP = PAPER_CONFIG
BATCH_SIZE = 64
DROPOUT = 0.2
DENSE_UNITS = 32
CFG = {"base_epochs": 100, "cold_epochs": 35, "warm_epochs": 10}


def set_global_seed(seed: int = SEED) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    for gpu in tf.config.list_physical_devices("GPU"):
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass


def resolve_data_path() -> Path:
    candidates = [
        Path("../data/raw/dataset_9_commodities_daily.csv"),
        Path("./data/raw/dataset_9_commodities_daily.csv"),
        Path("./dataset_9_commodities_daily.csv"),
        Path("/content/dataset_9_commodities_daily.csv"),
    ]
    for p in candidates:
        if p.exists():
            return p.resolve()
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for name in ["dataset_9_commodities_daily.csv", "commodities_9_daily.csv"]:
            exact = sorted(kaggle_input.rglob(name))
            if exact:
                return exact[0].resolve()
        for p in sorted(kaggle_input.rglob("*.csv")):
            try:
                cols = set(pd.read_csv(p, nrows=1).columns)
                if {"Date", "Copper", "WTI", "Gold", "Silver", "Aluminium"}.issubset(cols):
                    return p.resolve()
            except Exception:
                pass
    raise FileNotFoundError("Không tìm thấy dataset_9_commodities_daily.csv! Hãy Add Data trên Kaggle.")


OUT_DIR = Path("/kaggle/working/phase2d_80_results") if Path("/kaggle/working").exists() else Path("./phase2d_80_results")
CKPT_DIR = OUT_DIR / "ckpt_80"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = resolve_data_path()
set_global_seed(SEED)
print(f"DATA_PATH : {DATA_PATH}")
print(f"OUT_DIR   : {OUT_DIR}")
print(f"TF Version: {tf.__version__} | GPUs: {[g.name for g in tf.config.list_physical_devices('GPU')]}")

# Load & audit data
source = pd.read_csv(DATA_PATH)
dates = pd.to_datetime(source["Date"], errors="coerce")
data = source[["Copper"] + CANDIDATES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
data.insert(0, "Date", dates)
data = data.sort_values("Date").dropna().set_index("Date")
assert len(data) == 2710 and data.index.is_monotonic_increasing

def boundaries(n: int, ratio: float = TARGET_RATIO, val_fraction: float = VALIDATION_FRACTION):
    test_start = int(n * ratio)
    val_start = int(test_start * (1 - val_fraction))
    return val_start, test_start

fit_end_80, test_start_80 = boundaries(len(data), TARGET_RATIO, VALIDATION_FRACTION)
print(f"80/20 Split: fit_end={fit_end_80}, test_start={test_start_80}, n_test={len(data) - test_start_80}")


def dm_test_hac(actual: np.ndarray, pred_a: np.ndarray, pred_b: np.ndarray, loss: str = "L1", lag: int = 5):
    e_a = actual - pred_a
    e_b = actual - pred_b
    d = (np.abs(e_a) - np.abs(e_b)) if loss.upper() == "L1" else ((e_a ** 2) - (e_b ** 2))
    T = len(d)
    d_bar = float(np.mean(d))
    d_c = d - d_bar
    gamma_0 = float(np.dot(d_c, d_c) / T)
    var_hac = gamma_0
    for k in range(1, min(lag + 1, T)):
        w = 1.0 - k / (lag + 1.0)
        gamma_k = float(np.dot(d_c[k:], d_c[:-k]) / T)
        var_hac += 2.0 * w * gamma_k
    var_hac = max(var_hac, 1e-18)
    dm_raw = d_bar / math.sqrt(var_hac / T)
    hln = math.sqrt((T + 1 - 2 * 1 + 0) / T)
    dm_stat = dm_raw * hln
    p_val = float(2.0 * stats.t.sf(abs(dm_stat), df=T - 1))
    return float(dm_stat), float(p_val), float(d_bar)


def make_windows(x_scaled: np.ndarray, y_scaled: np.ndarray, dates: pd.DatetimeIndex,
                 start: int, end: int, lookback: int = LOOKBACK):
    if start < lookback or end <= start:
        raise ValueError(f"Khoảng cửa sổ không hợp lệ: start={start}, end={end}, lookback={lookback}")
    X = np.stack([x_scaled[t - lookback:t] for t in range(start, end)]).astype("float32")
    y = y_scaled[start:end].astype("float32")
    target_dates = dates[start:end]
    return X, y, target_dates

def make_walk_forward_fold(
    data: pd.DataFrame,
    fit_start: int,
    origin: int,
    block_end: int,
    features: list[str],
    lookback: int = LOOKBACK,
    target_mode: str = "direct",
):
    """Tạo 1 bước Walk-Forward (direct hoặc residual theo chuẩn Khoa/Công: delta trên thang MinMax của Copper)."""
    assert 0 <= fit_start < origin < block_end <= len(data), "Thứ tự chỉ số không hợp lệ."
    assert origin - fit_start > lookback + 10, "Cửa sổ huấn luyện quá ngắn so với lookback."
    if target_mode not in ("direct", "residual"):
        raise ValueError(f"target_mode không hợp lệ: {target_mode}")

    fit_df = data.iloc[fit_start:origin]
    x_scaler = MinMaxScaler().fit(fit_df[features])
    y_scaler = MinMaxScaler().fit(fit_df[["Copper"]])

    visible = data.iloc[fit_start:block_end]
    x_vis = x_scaler.transform(visible[features]).astype("float32")
    y_price_vis = y_scaler.transform(visible[["Copper"]]).ravel().astype("float32")

    local_origin = origin - fit_start
    local_end = block_end - fit_start

    X_train, y_train_price, train_dates = make_windows(x_vis, y_price_vis, visible.index, lookback, local_origin, lookback)
    X_eval, y_eval_price, eval_dates = make_windows(x_vis, y_price_vis, visible.index, local_origin, local_end, lookback)

    prev_train_scaled = y_price_vis[lookback - 1:local_origin - 1].astype("float32")
    prev_eval_scaled = y_price_vis[local_origin - 1:local_end - 1].astype("float32")

    if target_mode == "direct":
        y_train = y_train_price
        y_eval = y_eval_price
    else:
        # Chuẩn Khoa/Công: mạng học delta_y_t = y_scaled(t) - y_scaled(t-1), giá trị 0 tương ứng đứng yên
        y_train = (y_train_price - prev_train_scaled).astype("float32")
        y_eval = (y_eval_price - prev_eval_scaled).astype("float32")

    # Kiểm định bất biến chống rò rỉ dữ liệu
    assert train_dates.max() < eval_dates.min(), "Rò rỉ ngày giữa tập train và tập dự báo!"
    assert len(eval_dates) == (block_end - origin), "Thiếu mẫu dự báo trong block!"

    return {
        "X_train": X_train,
        "y_train": y_train,
        "train_dates": train_dates,
        "X_eval": X_eval,
        "y_eval_scaled": y_eval,
        "eval_dates": eval_dates,
        "y_eval_prev_scaled": prev_eval_scaled,
        "y_eval_actual": data["Copper"].iloc[origin:block_end].to_numpy(dtype=float),
        "y_eval_prev": data["Copper"].shift(1).iloc[origin:block_end].to_numpy(dtype=float),
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,
        "target_mode": target_mode,
    }

def compute_metrics(actual: np.ndarray, predicted: np.ndarray, prev_actual: np.ndarray,
                    y_train_for_mase: np.ndarray | None = None) -> dict:
    actual = np.asarray(actual, dtype=float).ravel()
    predicted = np.asarray(predicted, dtype=float).ravel()
    prev_actual = np.asarray(prev_actual, dtype=float).ravel()

    assert actual.shape == predicted.shape == prev_actual.shape
    assert np.isfinite(actual).all() and np.isfinite(predicted).all()

    mae = float(mean_absolute_error(actual, predicted))
    mse = float(mean_squared_error(actual, predicted))
    rmse = float(math.sqrt(mse))
    mape = float(np.mean(np.abs((actual - predicted) / np.clip(np.abs(actual), 1e-8, None))) * 100.0)
    naive_mae = float(mean_absolute_error(actual, prev_actual))

    if y_train_for_mase is not None and len(y_train_for_mase) > 1:
        in_sample_naive_mae = float(np.mean(np.abs(np.diff(y_train_for_mase))))
        mase = mae / max(in_sample_naive_mae, 1e-8)
    else:
        mase = mae / max(naive_mae, 1e-8)
    rel_mae_vs_naive = mae / max(naive_mae, 1e-8)

    d_act = actual - prev_actual
    d_pred = predicted - prev_actual
    if np.allclose(d_pred, 0.0):
        da_pct = float("nan")
        delta_corr = 0.0
        delta_std_ratio = 0.0
    else:
        nz = d_act != 0
        da_pct = float(np.mean(np.sign(d_pred[nz]) == np.sign(d_act[nz])) * 100.0) if nz.any() else float("nan")
        delta_corr = float(np.corrcoef(d_act, d_pred)[0, 1]) if (np.std(d_pred) > 1e-12 and np.std(d_act) > 1e-12) else 0.0
        delta_std_ratio = float(np.std(d_pred) / max(np.std(d_act), 1e-12))

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MSE_price2": mse,
        "MASE": float(mase),
        "Rel_MAE_Naive": float(rel_mae_vs_naive),
        "MAPE_pct": mape,
        "R2": float(r2_score(actual, predicted)) if len(actual) > 1 else float("nan"),
        "DA_pct": da_pct,
        "Delta_Corr": delta_corr,
        "Delta_Std_Ratio": delta_std_ratio,
    }


def build_lstm_model(n_features: int, lookback: int = LOOKBACK,
                     hp: dict = ACTIVE_HP, lr_multiplier: float = 1.0) -> tf.keras.Model:
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(lookback, n_features)),
        tf.keras.layers.LSTM(int(hp["lstm_units_1"]), activation="tanh",
                             recurrent_activation="sigmoid", return_sequences=True),
        tf.keras.layers.LSTM(int(hp["lstm_units_2"]), activation="tanh",
                             recurrent_activation="sigmoid"),
        tf.keras.layers.Dense(DENSE_UNITS, activation="relu"),
        tf.keras.layers.Dropout(DROPOUT),
        tf.keras.layers.Dense(1, activation="linear"),
    ])
    lr = float(hp["learning_rate"]) * lr_multiplier
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss="mse")
    return model

def fit_and_predict_fold(
    fold: dict,
    hp: dict = ACTIVE_HP,
    epochs: int = 30,
    prev_weights=None,
    update_mode: str = "cold_start",
    seed: int = SEED,
):
    tf.keras.backend.clear_session()
    set_global_seed(seed)

    n_features = fold["X_train"].shape[-1]
    use_warm = (update_mode == "warm_start") and (prev_weights is not None)
    model = build_lstm_model(n_features, LOOKBACK, hp, lr_multiplier=0.5 if use_warm else 1.0)
    if use_warm:
        model.set_weights(prev_weights)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="loss", patience=7, restore_best_weights=True, min_delta=1e-6
        )
    ]
    t0 = time.perf_counter()
    model.fit(
        fold["X_train"],
        fold["y_train"],
        epochs=epochs,
        batch_size=BATCH_SIZE,
        shuffle=False,
        verbose=0,
        callbacks=callbacks,
    )
    train_sec = time.perf_counter() - t0

    # Gọi trực tiếp model(..., training=False) giúp tránh retracing warning & nhanh hơn model.predict()
    pred_out = model(fold["X_eval"], training=False).numpy().ravel()
    if fold.get("target_mode", "direct") == "residual":
        pred_price_scaled = (fold["y_eval_prev_scaled"] + pred_out).reshape(-1, 1)
    else:
        pred_price_scaled = pred_out.reshape(-1, 1)

    pred_price = fold["y_scaler"].inverse_transform(pred_price_scaled).ravel()
    new_weights = model.get_weights()

    del model
    tf.keras.backend.clear_session()
    gc.collect()
    return pred_price, train_sec, new_weights


def run_forecasting_strategy(
    data: pd.DataFrame,
    strategy: str,
    eval_start: int,
    eval_end: int,
    features: list[str],
    hp: dict = ACTIVE_HP,
    window_size: int = 504,
    stride: int = 20,
    update_mode: str = "cold_start",
    base_epochs: int = CFG["base_epochs"],
    cold_epochs: int = CFG["cold_epochs"],
    warm_epochs: int = CFG["warm_epochs"],
    static_fit_end: int | None = None,
    seed: int = SEED,
    target_mode: str = "direct",
):
    """Chạy 1 chiến lược dự báo trên đoạn [eval_start, eval_end) không rò rỉ dữ liệu."""
    if strategy == "naive_persistence":
        dates = data.index[eval_start:eval_end]
        actual = data["Copper"].iloc[eval_start:eval_end].to_numpy(dtype=float)
        prev = data["Copper"].shift(1).iloc[eval_start:eval_end].to_numpy(dtype=float)
        df_pred = pd.DataFrame({
            "Date": dates,
            "actual": actual,
            "predicted": prev,
            "prev_actual": prev,
            "error": prev - actual,
            "abs_error": np.abs(prev - actual),
            "sq_error": (prev - actual) ** 2,
            "retrain_origin": eval_start,
        })
        m = compute_metrics(actual, prev, prev, data["Copper"].iloc[:eval_start].to_numpy(dtype=float))
        m.update({"target_mode": target_mode, "strategy": strategy, "n_retrains": 0, "total_train_sec": 0.0, "sec_per_retrain": 0.0})
        return m, df_pred

    if strategy in ("static_train_only", "static_train_val"):
        fit_end = static_fit_end if (strategy == "static_train_only" and static_fit_end is not None) else eval_start
        fold = make_walk_forward_fold(
            data=data, fit_start=0, origin=fit_end, block_end=eval_end, features=features, lookback=LOOKBACK, target_mode=target_mode
        )
        # Nếu fit_end < eval_start (static_train_only), cắt đúng đoạn [eval_start, eval_end) sau khi predict
        pred_all, train_sec, _ = fit_and_predict_fold(
            fold, hp=hp, epochs=base_epochs, prev_weights=None, update_mode="cold_start", seed=seed
        )
        offset = eval_start - fit_end
        dates = fold["eval_dates"][offset:]
        actual = fold["y_eval_actual"][offset:]
        prev = fold["y_eval_prev"][offset:]
        pred = pred_all[offset:]

        df_pred = pd.DataFrame({
            "Date": dates,
            "actual": actual,
            "predicted": pred,
            "prev_actual": prev,
            "error": pred - actual,
            "abs_error": np.abs(pred - actual),
            "sq_error": (pred - actual) ** 2,
            "retrain_origin": fit_end,
        })
        m = compute_metrics(actual, pred, prev, data["Copper"].iloc[:fit_end].to_numpy(dtype=float))
        m.update({"target_mode": target_mode, "strategy": strategy, "n_retrains": 1, "total_train_sec": train_sec, "sec_per_retrain": train_sec})
        return m, df_pred

    if strategy in ("expanding_window", "rolling_window"):
        origins = list(range(eval_start, eval_end, stride))
        all_frames = []
        total_sec = 0.0
        prev_weights = None

        for step_idx, origin in enumerate(origins):
            block_end = min(origin + stride, eval_end)
            fit_start = 0 if strategy == "expanding_window" else max(0, origin - window_size)
            fold = make_walk_forward_fold(
                data=data, fit_start=fit_start, origin=origin, block_end=block_end, features=features, lookback=LOOKBACK, target_mode=target_mode
            )
            epochs = base_epochs if step_idx == 0 else (warm_epochs if update_mode == "warm_start" else cold_epochs)
            pred_block, sec, prev_weights = fit_and_predict_fold(
                fold,
                hp=hp,
                epochs=epochs,
                prev_weights=prev_weights,
                update_mode=update_mode,
                seed=seed + step_idx,
            )
            total_sec += sec
            all_frames.append(pd.DataFrame({
                "Date": fold["eval_dates"],
                "actual": fold["y_eval_actual"],
                "predicted": pred_block,
                "prev_actual": fold["y_eval_prev"],
                "error": pred_block - fold["y_eval_actual"],
                "abs_error": np.abs(pred_block - fold["y_eval_actual"]),
                "sq_error": (pred_block - fold["y_eval_actual"]) ** 2,
                "retrain_origin": origin,
            }))

        df_pred = pd.concat(all_frames, ignore_index=True)
        assert len(df_pred) == (eval_end - eval_start)
        m = compute_metrics(
            df_pred["actual"].to_numpy(),
            df_pred["predicted"].to_numpy(),
            df_pred["prev_actual"].to_numpy(),
            data["Copper"].iloc[:eval_start].to_numpy(dtype=float),
        )
        m.update({
            "target_mode": target_mode,
            "strategy": strategy,
            "n_retrains": len(origins),
            "total_train_sec": total_sec,
            "sec_per_retrain": total_sec / max(1, len(origins)),
        })
        return m, df_pred

    raise ValueError(f"Unknown strategy: {strategy}")


DATA_PATH : /kaggle/input/datasets/lancelot71000/dataset-9-commodities-daily/dataset_9_commodities_daily.csv
OUT_DIR   : /kaggle/working/phase2d_80_results
TF Version: 2.21.0 | GPUs: ['/physical_device:GPU:0', '/physical_device:GPU:1']
80/20 Split: fit_end=1951, test_start=2168, n_test=542


In [2]:
import json

# Reference Naive baseline at 80/20
n_m, _ = run_forecasting_strategy(data, "naive_persistence", test_start_80, len(data), FEATURES_4, seed=SEED)
naive_mae = float(n_m["MAE"])
naive_rmse = float(n_m["RMSE"])
print(f"Naive Persistence 80/20 (N={len(data)-test_start_80}): MAE={naive_mae:.6f}, RMSE={naive_rmse:.6f}")

total_jobs = len(FEATURE_LIST) * len(STRATEGIES) * len(SEEDS_MULTI)
done_cnt = 0
t_global = time.perf_counter()

rows = []
all_preds = []

for fs_label, feat_cols in FEATURE_LIST:
    for strat in STRATEGIES:
        for s_val in SEEDS_MULTI:
            tag = f"r80__{fs_label}__{strat}__s{s_val}"
            jp = CKPT_DIR / f"{tag}.json"
            cp = CKPT_DIR / f"pred_{tag}.csv"

            if jp.exists() and cp.exists():
                meta = json.loads(jp.read_text(encoding="utf-8"))
                df_pred = pd.read_csv(cp)
            else:
                m, df_pred = run_forecasting_strategy(
                    data=data,
                    strategy=strat,
                    eval_start=test_start_80,
                    eval_end=len(data),
                    features=feat_cols,
                    window_size=BEST_W,
                    stride=BEST_S,
                    update_mode=BEST_UPDATE_MODE,
                    static_fit_end=fit_end_80,
                    seed=s_val,
                    target_mode="residual",
                )
                act = df_pred["actual"].to_numpy(dtype=float)
                prd = df_pred["predicted"].to_numpy(dtype=float)
                prv = df_pred["prev_actual"].to_numpy(dtype=float)
                dm1, p1, _ = dm_test_hac(act, prd, prv, loss="L1", lag=5)
                dm2, p2, _ = dm_test_hac(act, prd, prv, loss="L2", lag=5)

                df_pred = df_pred.copy()
                df_pred["ratio"] = TARGET_RATIO
                df_pred["feature_set"] = fs_label
                df_pred["strategy"] = strat
                df_pred["seed"] = s_val

                meta = {
                    "ratio": float(TARGET_RATIO),
                    "n_test": int(len(df_pred)),
                    "feature_set": fs_label,
                    "strategy": strat,
                    "target_mode": "residual",
                    "seed": int(s_val),
                    "MAE": float(m["MAE"]),
                    "RMSE": float(m["RMSE"]),
                    "MSE_price2": float(m["MSE_price2"]),
                    "MASE": float(m["MASE"]),
                    "Rel_MAE_Naive": float(m["Rel_MAE_Naive"]),
                    "MAPE_pct": float(m["MAPE_pct"]),
                    "R2": float(m["R2"]),
                    "DA_pct": float(m["DA_pct"]),
                    "Delta_Corr": float(m["Delta_Corr"]),
                    "Delta_Std_Ratio": float(m["Delta_Std_Ratio"]),
                    "Delta_MAE_vs_Naive_pct": (float(m["MAE"]) / naive_mae - 1.0) * 100.0,
                    "Delta_RMSE_vs_Naive_pct": (float(m["RMSE"]) / naive_rmse - 1.0) * 100.0,
                    "DM_HAC_L1": dm1,
                    "DM_HAC_L1_p": p1,
                    "DM_HAC_L2": dm2,
                    "DM_HAC_L2_p": p2,
                    "n_retrains": int(m["n_retrains"]),
                    "total_train_sec": float(m["total_train_sec"]),
                    "source": "kaggle_gpu",
                }
                df_pred.to_csv(cp, index=False)
                jp.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")

            rows.append(meta)
            all_preds.append(df_pred)
            done_cnt += 1
            elapsed = time.perf_counter() - t_global
            eta = (elapsed / done_cnt) * (total_jobs - done_cnt) / 60.0
            print(
                f"[{done_cnt:2d}/{total_jobs}] {fs_label:15s} | {strat:16s} | s={s_val} "
                f"| MAE={meta['MAE']:.5f} | DM_p={meta['DM_HAC_L1_p']:.4f} | wall={elapsed/60:.1f}m | ETA={eta:.1f}m",
                flush=True,
            )

df_metrics = pd.DataFrame(rows)
df_preds = pd.concat(all_preds, ignore_index=True)
assert len(df_metrics) == 60
assert df_metrics.isna().sum().sum() == 0
assert len(df_preds) == 60 * 542

df_metrics.to_csv(OUT_DIR / "phase2d_80_metrics.csv", index=False)
df_preds.to_csv(OUT_DIR / "phase2d_80_predictions.csv", index=False)

sum_rows = []
for fs_label, _ in FEATURE_LIST:
    for strat in STRATEGIES:
        sub = df_metrics[(df_metrics["feature_set"] == fs_label) & (df_metrics["strategy"] == strat)]
        sub_p = df_preds[(df_preds["feature_set"] == fs_label) & (df_preds["strategy"] == strat)]
        ens = sub_p.groupby("Date", as_index=False).agg(
            actual=("actual", "first"),
            prev_actual=("prev_actual", "first"),
            predicted=("predicted", "mean"),
        ).sort_values("Date")
        act = ens["actual"].to_numpy(dtype=float)
        prd = ens["predicted"].to_numpy(dtype=float)
        prv = ens["prev_actual"].to_numpy(dtype=float)
        ens_mae = float(np.mean(np.abs(act - prd)))
        ens_rmse = float(np.sqrt(np.mean((act - prd) ** 2)))
        ens_dm, ens_p, _ = dm_test_hac(act, prd, prv, loss="L1", lag=5)
        sum_rows.append({
            "ratio": TARGET_RATIO,
            "feature_set": fs_label,
            "strategy": strat,
            "n_seeds": int(sub["seed"].nunique()),
            "naive_MAE": naive_mae,
            "MAE_mean": float(sub["MAE"].mean()),
            "MAE_std": float(sub["MAE"].std(ddof=1)),
            "RMSE_mean": float(sub["RMSE"].mean()),
            "DA_mean": float(sub["DA_pct"].mean()),
            "DeltaMAE_vs_Naive_mean_pct": float(sub["Delta_MAE_vs_Naive_pct"].mean()),
            "Ensemble_5seed_MAE": ens_mae,
            "Ensemble_5seed_RMSE": ens_rmse,
            "Ensemble_vs_Naive_pct": (ens_mae / naive_mae - 1.0) * 100.0,
            "Ensemble_DM_HAC_L1": ens_dm,
            "Ensemble_DM_HAC_p": ens_p,
            "seeds_beat_naive": int((sub["MAE"] < naive_mae).sum()),
            "seeds_DM_win_5pct": int(((sub["DM_HAC_L1"] < 0) & (sub["DM_HAC_L1_p"] < 0.05)).sum()),
            "seeds_DM_loss_5pct": int(((sub["DM_HAC_L1"] > 0) & (sub["DM_HAC_L1_p"] < 0.05)).sum()),
        })

df_summary = pd.DataFrame(sum_rows)
df_summary.to_csv(OUT_DIR / "phase2d_80_summary.csv", index=False)
print("Saved phase2d_80_metrics.csv, phase2d_80_predictions.csv, phase2d_80_summary.csv cleanly!")


Naive Persistence 80/20 (N=542): MAE=0.069001, RMSE=0.108000
[ 1/60] FS01_CU         | expanding_window | s=41 | MAE=0.06922 | DM_p=0.1205 | wall=2.1m | ETA=125.4m
[ 2/60] FS01_CU         | expanding_window | s=42 | MAE=0.06934 | DM_p=0.0751 | wall=4.4m | ETA=126.6m
[ 3/60] FS01_CU         | expanding_window | s=43 | MAE=0.06939 | DM_p=0.1676 | wall=6.7m | ETA=126.8m
[ 4/60] FS01_CU         | expanding_window | s=44 | MAE=0.06922 | DM_p=0.2159 | wall=9.1m | ETA=127.4m
[ 5/60] FS01_CU         | expanding_window | s=45 | MAE=0.06932 | DM_p=0.0898 | wall=11.5m | ETA=126.5m
[ 6/60] FS01_CU         | rolling_window   | s=41 | MAE=0.06950 | DM_p=0.0987 | wall=13.4m | ETA=120.9m
[ 7/60] FS01_CU         | rolling_window   | s=42 | MAE=0.06961 | DM_p=0.0255 | wall=15.4m | ETA=116.8m
[ 8/60] FS01_CU         | rolling_window   | s=43 | MAE=0.06945 | DM_p=0.0239 | wall=17.6m | ETA=114.4m
[ 9/60] FS01_CU         | rolling_window   | s=44 | MAE=0.06954 | DM_p=0.0119 | wall=19.8m | ETA=112.0m
[10/60]

In [3]:
display(df_summary.round(5))

zip_out = Path("/kaggle/working/phase2d_80_results") if Path("/kaggle/working").exists() else Path("./phase2d_80_results_zip")
archive_path = shutil.make_archive(str(zip_out), "zip", root_dir=OUT_DIR)
print(f"\nHOÀN TẤT 60/60 (80/20)! Tải tệp ZIP này về máy: {archive_path}")


,ratio,feature_set,strategy,n_seeds,naive_MAE,MAE_mean,MAE_std,RMSE_mean,DA_mean,DeltaMAE_vs_Naive_mean_pct,Ensemble_5seed_MAE,Ensemble_5seed_RMSE,Ensemble_vs_Naive_pct,Ensemble_DM_HAC_L1,Ensemble_DM_HAC_p,seeds_beat_naive,seeds_DM_win_5pct,seeds_DM_loss_5pct
0,0.8,FS01_CU,expanding_window,5,0.069,0.06930,0.00007,0.10816,48.48148,0.42884,0.06926,0.10812,0.37354,1.56806,0.11745,0,0,0
1,0.8,FS01_CU,rolling_window,5,0.069,0.06952,0.00006,0.10857,47.96296,0.75116,0.06946,0.10852,0.65755,2.23953,0.02553,0,0,4
2,0.8,FEATURES_4,expanding_window,5,0.069,0.07004,0.00084,0.10948,51.55556,1.50229,0.06932,0.10889,0.46278,0.64100,0.52180,0,0,2
3,0.8,FEATURES_4,rolling_window,5,0.069,0.07177,0.00104,0.11051,51.07407,4.01739,0.07098,0.10958,2.86246,2.34378,0.01945,0,0,4
4,0.8,FEATURES_3,expanding_window,5,0.069,0.07001,0.00069,0.10965,50.59259,1.46297,0.06960,0.10922,0.87310,1.43668,0.15139,0,0,3
5,0.8,FEATURES_3,rolling_window,5,0.069,0.07077,0.00099,0.11077,52.29630,2.56698,0.06984,0.10966,1.21183,1.52395,0.12811,0,0,2
6,0.8,FS04_METALS_CU,expanding_window,5,0.069,0.07054,0.00097,0.11005,51.37037,2.22317,0.06972,0.10932,1.03758,1.29727,0.19509,0,0,4
7,0.8,FS04_METALS_CU,rolling_window,5,0.069,0.07110,0.00034,0.11031,51.92593,3.04848,0.07048,0.10965,2.13955,2.09827,0.03634,0,0,5
8,0.8,FS05_METALS_EXO,expanding_window,5,0.069,0.06986,0.00074,0.10903,49.33333,1.23899,0.06949,0.10861,0.71122,1.42078,0.15596,0,0,1
9,0.8,FS05_METALS_EXO,rolling_window,5,0.069,0.07084,0.00062,0.11034,50.70370,2.65989,0.07011,0.10956,1.60990,2.33134,0.02010,0,0,5



HOÀN TẤT 60/60 (80/20)! Tải tệp ZIP này về máy: /kaggle/working/phase2d_80_results.zip
